In [1]:
import numpy as np
from pyscf import gto, scf, lo, mp, cc
mol = gto.Mole()
mol.verbose = 4
mol.atom = '''
O   -1.485163346097   -0.114724564047    0.000000000000
H   -1.868415346097    0.762298435953    0.000000000000
H   -0.533833346097    0.040507435953    0.000000000000
O    1.416468653903    0.111264435953    0.000000000000
H    1.746241653903   -0.373945564047   -0.758561000000
H    1.746241653903   -0.373945564047    0.758561000000
'''
mol.basis = 'cc-pvdz'
mol.precision = 1e-10
mol.build()
mf = scf.RHF(mol).density_fit()
mf.kernel()

# frozen = 0
mymp = mp.MP2(mf).set_frozen()
mymp.kernel()
efull_mp2 = mymp.e_corr
print(f'MP2 Corr = {efull_mp2:.8f}')

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()
efull_ccsd = mycc.e_corr
print(f'CCSD Corr = {efull_ccsd:.8f}')

efull_t = mycc.ccsd_t()
efull_ccsd_t = efull_ccsd + efull_t
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}')

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-35-generic', version='#35~24.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue May 26 19:30:42 UTC 2', machine='x86_64')  Threads 16
Python 3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:20:58) [GCC 14.3.0]
numpy 2.4.4  scipy 1.17.1  h5py 3.16.0
Date: Sun Jun 14 17:28:27 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 6
[INPUT] num. electrons = 20
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y               

In [4]:
print(f'MP2 Corr = {efull_mp2:.8f}')
print(f'CCSD Corr = {efull_ccsd:.8f}')
print(f'CCSD(T) Corr = {efull_ccsd_t:.8f}') 

MP2 Corr = -0.40609899
CCSD Corr = -0.42461940
CCSD(T) Corr = -0.43105819


In [56]:
import jax
jax.config.update("jax_enable_x64", True)
import opt_einsum as oe
import jax.numpy as jnp
from afqmc.lno_afqmc import prep, lno_afqmc
from pyscf import lib
from pyscf.lno import lnoccsd, ulnoccsd
from jax import random
from collections.abc import Iterable
from afqmc.lno_afqmc import prep, tools
from afqmc.lno_afqmc import mod_lnoccsd

from functools import reduce

from functools import partial
import time, gc, pickle
import os

In [6]:
from afqmc.lno_afqmc import lno_afqmc, tools
iao_coeff, iao_frag_list, atm_center = tools.iao_localization(mf)

In [7]:
options = {
           'eql_time': 10,
           'n_blocks': 50,
           'n_walkers': 300,
           'mix_precision': True,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'upt2ccsd',
           }

mf = mf
lo_coeff = iao_coeff 
frag_lolist = iao_frag_list
nfrozen = mycc.frozen
thresh = 1e-5
qmc_options = options
chol_cut = 1e-5 
target_sto_error = 1e-3
run_frag_list = [0]
atom_group = None
plot_las = False

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

if spin_type == "unrestricted":
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mf = mlno._scf
else:
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)

mlno.lno_thresh = [thresh*10, thresh]
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag_tot = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag_tot)

frag_lolist = [frag_lolist[i] for i in run_frag_list]
nfrag_run = len(frag_lolist)
print(f"Run {nfrag_run} fragments out of {nfrag_tot} total fragments")

lno_pct_occ = [None, None]
lno_norb = [[None, None]] * nfrag_run

seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                        shape=(nfrag_run,), 
                        minval=0, 
                        maxval=100*nfrag_run
                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag_run
las_size = [None]*nfrag_run
# las_size = np.zeros(nfrag, dtype='int32')
lno_emp2 = np.zeros(nfrag_run, dtype='float64')
lno_ecc  = np.zeros(nfrag_run, dtype='float64')
lno_eqmc = np.zeros(nfrag_run, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')
ccsd_time = np.zeros(nfrag_run, dtype='float64')
qmc_time = np.zeros(nfrag_run, dtype='float64')

mol = mf.mol
print(lno_thresh)
print(lno_pct_occ)
print(lno_norb)

# Loop over fragment
for ifrag, loidx in enumerate(frag_lolist):
    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {run_frag_list[ifrag]+1}/({nfrag_run},{nfrag_tot}) "
    print(msg.center(width, '='))
    if atom_group is not None:
        loc_ctr = f"{atom_group[run_frag_list[ifrag]]}"
        print(f"Center Atom {loc_ctr}")
    else:
        loc_ctr = None
    
    print(f"PySCF NumPy Threads = {lib.num_threads()}")

    if spin_type == "unrestricted":
        orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
        lno_param = [
            [
                {
                    'thresh': (
                        lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                        else lno_thresh[i]
                    ),
                    'pct_occ': (
                        lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                        else lno_pct_occ[i]
                    ),
                    'norb': (
                        lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                        else lno_norb[ifrag][i]
                    ),
                } for i in [0, 1]
            ] for s in range(2)
        ]
    else:
        orbloc = lo_coeff[:,loidx]
        lno_param = [{
            'thresh': lno_thresh[i],
            'pct_occ': lno_pct_occ[i],
            'norb': lno_norb[ifrag][i]
            } for i in [0,1]]

    # M = <orbloc|canactocc> (M^dagger M)u = eu
    # u|canactocc> => orbtial in/out the space spanned by |orbloc>
    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = mlno.make_las(eris, orbloc, lno_type, lno_param)

    mo_occ = mlno.mo_occ

    if spin_type == "unrestricted":
        if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
            lno_elec_type = 'alpha'
        elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
            lno_elec_type = 'beta'
        else:
            lno_elec_type = 'mixed'
        print(f'LNO-Frgament Spin Type = {lno_elec_type}')

        if loc_ctr is None:
            ao_max_a = prep.ao_comp(mf, orbloc[0])
            ao_max_b = prep.ao_comp(mf, orbloc[1])
            loc_ctr = ao_max_a + ao_max_b
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
        occidxa = mo_occ[0] > 1e-10
        occidxb = mo_occ[1] > 1e-10
        moidxa, moidxb = maskact
        nactocc_a = int(np.sum(moidxa & occidxa))
        nactvir_a = int(np.sum(moidxa & ~occidxa))
        nactocc_b = int(np.sum(moidxb & occidxb))
        nactvir_b = int(np.sum(moidxb & ~occidxb))
        nactocc = [nactocc_a, nactocc_b]
        nactvir = [nactvir_a, nactvir_b]
        lno_active_a = np.array([i for i in range(mol.nao) if i not in lno_frozen[0]])
        lno_active_b = np.array([i for i in range(mol.nao) if i not in lno_frozen[1]])
        lno_active = [lno_active_a, lno_active_b]
        lno_tot = [len(lno_active_a), len(lno_active_b)]
        # print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    else:
        print(f'LNO-Frgament Spin Type = restricted')
        if loc_ctr is None:
            loc_ctr = prep.ao_comp(mf, orbloc)
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
        lno_active = np.array([i for i in range(mol.nao) if i not in lno_frozen])
        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        lno_tot = len(lno_active)
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    
    if plot_las:
        tools.plot_density(mol, orbloc, lno_coeff, lno_active, spin_type, idx = run_frag_list[ifrag]+1)

    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 =\
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
    time1 = time.perf_counter()
    lnocc_time = time1 - time0

    print(f'LNO-MP2 Orbital Energy:  {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')
    print(f"LNO-CCSD time:           {lnocc_time:.2f} s")

    # las_center[ifrag] = loc_ctr
    # las_size[ifrag] = lno_tot
    # lno_emp2[ifrag] = eorb_mp2
    # lno_ecc[ifrag] = eorb_cc
    # ccsd_time[ifrag] = lnocc_time

    # # project onto center lo space
    # # <lno_actocc|orbloc> <orbloc|lno_actocc>
    # if spin_type == "unrestricted":
    #     prjlo = [uocc_loc[0] @ uocc_loc[0].T.conj(),
    #                 uocc_loc[1] @ uocc_loc[1].T.conj()]
    #     qmc_options["trial"] = trial_base
    #     if 'ad' not in trial_base:
    #         if lno_elec_type == 'alpha':
    #             qmc_options["trial"] += '_alpha'
    #         elif lno_elec_type == 'beta':
    #             qmc_options["trial"] += '_beta'
    # else:
    #     prjlo = uocc_loc @ uocc_loc.T.conj()

    # qmc_options["seed"] = seeds[ifrag]
    # prep.prep_afqmc_integral(
    #     mf,
    #     lno_coeff,
    #     t1,
    #     t2,
    #     lno_frozen,
    #     prjlo,
    #     qmc_options,
    #     chol_cut=chol_cut
    #     )
    
    # lno_afqmc.run_lnoafqmc(qmc_options)

Run 1 fragments out of 6 total fragments
[0.0001, 1e-05]
[None, None]
[[None, None]]


======================= restricted LNO-FRAGMENT 1/(1,6) ========================
PySCF NumPy Threads = 16
LNO-Frgament Spin Type = restricted
LNO Center 0 O 1s    
LAS occupied orbitals:  7
LAS virtual orbitals:   28
LAS total size:         35
LNO-MP2 Orbital Energy:  -0.17003705
LNO-CCSD Orbital Energy: -0.17752155
LNO-CCSD time:           0.15 s


In [10]:
nocc = np.count_nonzero(mf.mo_occ)
orboccfrz_core, orbocc, orbvir, orbvirfrz_core = mlno.split_mo_coeff()
print((orboccfrz_core-mf.mo_coeff[:,:mycc.frozen]).max())
print((orbocc-mf.mo_coeff[:,mycc.frozen:nocc]).max())

0.0
0.0


In [23]:
iao_coeff[:,frag_lolist[0]].shape

(48, 5)

In [24]:
s1e = mf.get_ovlp()
lo2mo = iao_coeff[:,frag_lolist[0]].T @ s1e @ orbocc

In [64]:
def projection_construction(M, thresh=1e-10, thresh_act=None):
    r''' Given M_{mu,i} = <mu | i> the ovlp between two orthonormal basis, find
    the unitary rotation |j'> = u_ij |i> so that {|j'>} significantly ovlp with
    {|mu>}.

    Three subsets will be returned:
        active  : singular value >  thresh_act
        standby : singular value <= thresh_act but > thresh
        frozen  : singular value <= thresh
    '''
    n, m = M.shape
    e, u = np.linalg.eigh(np.dot(M.T.conj(), M))
    if thresh_act is None: thresh_act = thresh
    assert( thresh_act >= thresh )
    mask_act = abs(e) > thresh_act
    mask_std = np.logical_and(abs(e) > thresh, ~mask_act)
    mask_frz = abs(e) <= thresh
    return u[:,mask_act], u[:,mask_std], u[:,mask_frz]

In [74]:
@jax.jit
def _projection_core(M):
    # M^dagger M is Hermitian PSD; eigh runs on GPU
    m2 = jnp.dot(M.conj().T, M)
    e, u = jnp.linalg.eigh(m2)
    return e, u

def projection_construction_gpu(M, thresh=1e-10, thresh_act=None):
    r''' Given M_{mu,i} = <mu | i> the ovlp between two orthonormal basis, find
    the unitary rotation |j'> = u_ij |i> so that {|j'>} significantly ovlp with
    {|mu>}.
    Three subsets will be returned:
        active  : singular value >  thresh_act
        standby : singular value <= thresh_act but > thresh
        frozen  : singular value <= thresh
    '''
    if thresh_act is None:
        thresh_act = thresh
    assert thresh_act >= thresh

    M = np.asarray(M)
    thresh = np.asarray(thresh, M.real.dtype)
    thresh_act = np.asarray(thresh_act, M.real.dtype)

    e, u = _projection_core(M)
    
    ae = jnp.abs(e)
    mask_act = ae > thresh_act
    mask_std = (ae > thresh) & (~mask_act)
    mask_frz = ae <= thresh

    # Boolean indexing happens OUTSIDE jit (dynamic shapes).
    return u[:,mask_act], u[:,mask_std], u[:,mask_frz]

In [76]:
uocc_loc, uocc_std, uocc_orth = projection_construction(lo2mo, thresh_act=0.1)
uocc_loc_gpu, uocc_std_gpu, uocc_orth_gpu = projection_construction_gpu(lo2mo, thresh_act=0.1)

In [77]:
print((uocc_loc-uocc_loc_gpu).max())
print((uocc_std-uocc_std_gpu).max())
print((uocc_orth-uocc_orth_gpu).max())

1.5805654925591854e-13
3.0098146197587994e-12
0.6878131138918502


In [68]:
def sub_colspace(A, idx):
    if idx.size == 0:
        return np.zeros([A.shape[0],0])
    else:
        return A[:,idx]
    
def natorb_select(dm, orb, thresh, pct_occ=None, norb=None):
    e, u = np.linalg.eigh(dm)
    e = abs(e)
    order = np.argsort(e)[::-1]
    e = e[order]
    u = u[:,order]
    if norb is None:
        if pct_occ is None:
            nkeep = np.count_nonzero(e > thresh)
        else:
            nkeep = np.count_nonzero(np.cumsum(e)/np.sum(e) <= pct_occ)
    else:
        nkeep = min(max(norb, 0), e.size)

    idx  = np.arange(0,     nkeep,  dtype=int)
    idxc = np.arange(nkeep, e.size, dtype=int)
    orbx = np.dot(orb, u)
    orb1x = sub_colspace(orbx, idx)
    orb0x = sub_colspace(orbx, idxc)
    return orb1x, orb0x

In [69]:
def sub_colspace_gpu(A, idx):
    # idx is a contiguous arange in the caller, so a plain slice is cleaner and
    # keeps shapes static. Empty idx naturally yields an (A.shape[0], 0) array.
    if idx.size == 0:
        return jnp.zeros((A.shape[0], 0), dtype=A.dtype)
    return A[:, idx]


@jax.jit
def _natorb_core_gpu(dm, orb):
    # Hermitian eigendecomposition on GPU
    e, u = jnp.linalg.eigh(dm)
    e = jnp.abs(e)

    # descending order
    order = jnp.argsort(e)[::-1]
    e = e[order]
    u = u[:, order]

    orbx = jnp.dot(orb, u)
    return e, orbx


def natorb_select_gpu(dm, orb, thresh, pct_occ=None, norb=None):
    dm = jnp.asarray(dm)
    orb = jnp.asarray(orb)

    e, orbx = _natorb_core_gpu(dm, orb)

    # nkeep is data-dependent -> compute it, then concretize for the slice.
    if norb is None:
        if pct_occ is None:
            nkeep = int(jnp.count_nonzero(e > thresh))
        else:
            cum = jnp.cumsum(e) / jnp.sum(e)
            nkeep = int(jnp.count_nonzero(cum <= pct_occ))
    else:
        nkeep = min(max(norb, 0), int(e.size))

    # Contiguous split outside jit (dynamic boundary, static-per-call shapes).
    orb1x = orbx[:, :nkeep]
    orb0x = orbx[:, nkeep:]
    return orb1x, orb0x

In [70]:
def subspace_eigh(fock, orb):
    f = reduce(np.dot, (orb.T.conj(), fock, orb))
    if orb.shape[1] == 1:
        moe = np.array([f[0,0]])
    else:
        moe, u = np.linalg.eigh(f)
        orb = np.dot(orb, u)
    return moe, orb

@jax.jit
def subspace_eigh_gpu(fock, orb):
    # orb^H @ fock @ orb
    f = reduce(jnp.dot, (orb.conj().T, fock, orb))

    if orb.shape[1] == 1:
        # 1x1 block: eigenvalue is just the (real) diagonal element.
        moe = jnp.real(f[0, 0]).reshape(1)
        return moe, orb

    moe, u = jnp.linalg.eigh(f)
    orb = jnp.dot(orb, u)
    return moe, orb

In [79]:
s1e = mlno.s1e
orboccfrz_core, orbocc, orbvir, orbvirfrz_core = mlno.split_mo_coeff()
# moeocc, moevir = mlno.split_mo_energy()[1:3]
uocc_loc = orbloc.T.conj() @ s1e @ orbocc # <fraglo|mo_occ>
uocc_loc, uocc_std, uocc_orth = \
    projection_construction(uocc_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)

In [82]:
uocc_loc

array([[-4.12411866e-02,  4.72035569e-02,  2.60432001e-02,
         9.15365297e-15],
       [-6.64578616e-01,  5.85679682e-01,  4.54320607e-01,
         1.58149013e-13],
       [-2.45340581e-16,  2.26399316e-17, -2.80134312e-15,
         7.09304242e-03],
       [ 6.26187145e-01,  7.05542708e-01, -4.14986609e-02,
        -1.95779560e-14],
       [-2.11218838e-01, -1.69209990e-01, -1.77770273e-01,
        -5.74583038e-14],
       [-3.27499844e-01,  1.52344143e-01, -7.22456307e-01,
        -2.52876329e-13],
       [-1.12492406e-01,  3.24221935e-01, -4.87490527e-01,
        -1.67676426e-13],
       [ 3.92499127e-15,  3.92252939e-15, -3.47539561e-13,
         9.99974844e-01]])

In [ ]:
def _mp2_rdm1_occblksize(nocc, nvir, naux, n1, n2, M, dsize):
    r''' Estimate block size for the occupied index in MP2 rdm1 evaluation.

        Model:
            Assuming storing n1 copies of ([O]V|L) and n2 copies of ([O]V|[O]V), [O] is
            determined by solving a quadratic equation:
                (n2*V^2) * [O]^2 + (n1*V*X) * [O] - M = 0

        Args:
            nocc/nvir/naux : int
                Number of occ/vir/aux orbitals.
            n1/n2 : int
                Number of copies of tensors of size nvir*naux*occblksize and nvir^2*occblksize^2.
            M: float or int
                Available memory in terms how many numbers to store, i.e., mem_in_MB * 1e6/dsize.

        Return:
            occblksize (int)
            mem_peak (float) : peak memory (in MB)
    '''
    occblksize = max(1, min(nocc, int(np.floor((((n1*naux)**2 + 4*n2*M)**0.5 - n1*naux) /
                                                (2*n2*nvir)))))
    mem_peak = (occblksize * nvir*naux * n1 + occblksize**2 * nvir**2 * n2) * dsize/1e6
    return occblksize, mem_peak

def make_lo_rdm1_occ_1h(eris, moeocc, moevir, u):
    r''' Occupied MP2 density matrix with one localized hole

        Math:
            dm(i,j)
                = 2 * \sum_{k'ab} t2(ik'ab) ( 2*t2(jk'ab) - t2(jk'ba) )

        Args:
            eris : ERI object
                Provides `ovL` in the canonical MOs.
            u : np.ndarray
                Overlap between the canonical and localized occupied orbitals, i.e.,
                    u(i,i') = <i|i'>
    '''
    nocc, nvir, naux = eris.nocc, eris.nvir, eris.naux
    dtype = eris.dtype
    dsize = eris.dsize
    nOcc = u.shape[1]

    # determine occblksize
    mem_avail = eris.max_memory - lib.current_memory()[0]
    M = mem_avail * 0.7 * 1e6/dsize
    occblksize, mem_peak = _mp2_rdm1_occblksize(nocc,nvir,naux, 3, 3, M, dsize)
    if DEBUG_BLKSIZE: occblksize = max(1,nocc//2)
    logger.debug1(eris, 'make_lo_rdm1_occ_1h :  nocc = %d  nvir = %d  nOcc = %d  naux = %d  '
                  'occblksize = %d  peak mem = %.2f MB',
                  nocc, nvir, nOcc, naux, occblksize, mem_peak)

    moeOcc, u = subspace_eigh(np.diag(moeocc), u)
    eov = moeocc[:,None] - moevir
    eOv = moeOcc[:,None] - moevir

    dm = np.zeros((nocc,nocc), dtype=dtype)
    for Kbatch,(K0,K1) in enumerate(lib.prange(0,nOcc,occblksize)):
        KvL = eris.xform_occ(u[:,K0:K1])
        eKv = eOv[K0:K1]
        for ibatch,(i0,i1) in enumerate(lib.prange(0,nocc,occblksize)):
            ivL = eris.get_occ_blk(i0,i1)
            eiv = eov[i0:i1]
            eiKvv = lib.direct_sum('ia+Kb->iKab', eiv, eKv)
            t2iKvv = np.conj(lib.einsum('iax,Kbx->iKab', ivL, KvL)) / eiKvv
            ivL = None
            eiKvv = None
            for jbatch,(j0,j1) in enumerate(lib.prange(0,nocc,occblksize)):
                if jbatch == ibatch:
                    t2jKvv = t2iKvv
                else:
                    jvL = eris.get_occ_blk(j0,j1)
                    ejv = eov[j0:j1]
                    ejKvv = lib.direct_sum('ia+Kb->iKab', ejv, eKv)
                    t2jKvv = np.conj(lib.einsum('iax,Kbx->iKab', jvL, KvL)) / ejKvv
                    jvL = None
                    ejKvv = None

                dm[i0:i1,j0:j1] -= 4 * lib.einsum('iKab,jKab->ij', np.conj(t2iKvv), t2jKvv)
                dm[i0:i1,j0:j1] += 2 * lib.einsum('iKab,jKba->ij', np.conj(t2iKvv), t2jKvv)

                t2jKvv = None
            t2iKvv = None
        KvL = None

    return dm

In [88]:
from pyscf.lib import logger

def make_las(mlno, eris, orbloc, lno_type, lno_param):
    log = logger.new_logger(mlno)
    cput1 = (logger.process_clock(), logger.perf_counter())

    s1e = mlno.s1e

    orboccfrz_core, orbocc, orbvir, orbvirfrz_core = mlno.split_mo_coeff()
    moeocc, moevir = mlno.split_mo_energy()[1:3]

    ''' Projection of LO onto occ and vir
    '''
    uocc_loc = orbloc.T.conj() @ s1e @ orbocc # <fraglo|mo_occ>
    uocc_loc, uocc_std, uocc_orth = \
            projection_construction(uocc_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)
    if uocc_loc.shape[1] == 0:
        log.error('LOs do not overlap with occupied space. This could be caused '
                  'by either a bad fragment choice or too high of `lo_proj_thresh_active` '
                  '(current value: %s).', mlno.lo_proj_thresh_active)
        raise RuntimeError
    log.info('LO occ proj: %d active | %d standby | %d orthogonal',
             *[u.shape[1] for u in [uocc_loc,uocc_std,uocc_orth]])

    uvir_loc = reduce(np.dot, (orbloc.T.conj(), s1e, orbvir))
    uvir_loc, uvir_std, uvir_orth = \
            projection_construction(uvir_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)
    log.info('LO vir proj: %d active | %d standby | %d orthogonal',
             *[u.shape[1] for u in [uvir_loc,uvir_std,uvir_orth]])
    if uvir_loc.shape[1] == 0:
        uvir_loc = uvir_std = uvir_orth = None

    ''' LNO construction
    '''
    dmoo = mlno.make_lo_rdm1_occ(eris, moeocc, moevir, uocc_loc, uvir_loc, lno_type[0])
    if mlno._match_oldcode: dmoo *= 0.5 # TO MATCH OLD LNO CODE
    dmoo = reduce(np.dot, (uocc_orth.T.conj(), dmoo, uocc_orth))
    if lno_param[0]['norb'] is not None:
        lno_param[0]['norb'] -= uocc_loc.shape[1] + uocc_std.shape[1]
    uoccact_orth, uoccfrz_orth = natorb_select(dmoo, uocc_orth, **lno_param[0])
    orboccfrz = np.hstack((orboccfrz_core, np.dot(orbocc, uoccfrz_orth)))
    uoccact = subspace_eigh(np.diag(moeocc), np.hstack((uoccact_orth, uocc_std, uocc_loc)))[1]
    orboccact = np.dot(orbocc, uoccact)
    uoccact_loc = np.linalg.multi_dot((orboccact.T.conj(), s1e, orbloc))
    cput1 = log.timer_debug1('make_lo_rdm1_occ', *cput1)

    dmvv = mlno.make_lo_rdm1_vir(eris, moeocc, moevir, uocc_loc, uvir_loc, lno_type[1])
    if mlno._match_oldcode: dmvv *= 0.5 # TO MATCH OLD LNO CODE
    if uvir_orth is not None:
        dmvv = reduce(np.dot, (uvir_orth.T.conj(), dmvv, uvir_orth))
        if lno_param[1]['norb'] is not None:
            lno_param[1]['norb'] -= uvir_loc.shape[1] + uvir_std.shape[1]
        uviract_orth, uvirfrz_orth = natorb_select(dmvv, uvir_orth, **lno_param[1])
        orbvirfrz = np.hstack((np.dot(orbvir, uvirfrz_orth), orbvirfrz_core))
        uviract = subspace_eigh(np.diag(moevir), np.hstack((uviract_orth, uvir_std, uvir_loc)))[1]
        orbviract = np.dot(orbvir, uviract)
    else:
        orbviract, orbvirfrz = natorb_select(dmvv, orbvir, **lno_param[1])
        orbvirfrz = np.hstack((orbvirfrz, orbvirfrz_core))
        uviract = reduce(np.dot, (orbvir.T.conj(), s1e, orbviract))
        uviract = subspace_eigh(np.diag(moevir), uviract)[1]
        orbviract = np.dot(orbvir, uviract)
    cput1 = log.timer_debug1('make_lo_rdm1_vir', *cput1)

    ''' LAS construction
    '''
    orbfragall = [orboccfrz, orboccact, orbviract, orbvirfrz]
    orbfrag = np.hstack(orbfragall)
    norbfragall = np.asarray([x.shape[1] for x in orbfragall])
    locfragall = np.cumsum([0] + norbfragall.tolist()).astype(int)
    frzfrag = np.concatenate((
        np.arange(locfragall[0], locfragall[1]),
        np.arange(locfragall[3], locfragall[4]))).astype(int)
    frag_msg = '%d/%d Occ | %d/%d Vir | %d/%d MOs' % (
                    norbfragall[1], sum(norbfragall[:2]),
                    norbfragall[2], sum(norbfragall[2:4]),
                    sum(norbfragall[1:3]), sum(norbfragall)
                )
    if len(frzfrag) == 0:
        frzfrag = 0

    return orbfrag, frzfrag, uoccact_loc, frag_msg

def make_las_gpu(mlno, eris, orbloc, lno_type, lno_param):
    log = logger.new_logger(mlno)
    cput1 = (logger.process_clock(), logger.perf_counter())

    s1e = mlno.s1e

    orboccfrz_core, orbocc, orbvir, orbvirfrz_core = mlno.split_mo_coeff()
    moeocc, moevir = mlno.split_mo_energy()[1:3]

    ''' Projection of LO onto occ and vir
    '''
    uocc_loc = reduce(jnp.dot, (orbloc.T.conj(), s1e, orbocc)) # <fraglo|mo_occ>
    uocc_loc, uocc_std, uocc_orth = \
            projection_construction_gpu(uocc_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)
    if uocc_loc.shape[1] == 0:
        log.error('LOs do not overlap with occupied space. This could be caused '
                  'by either a bad fragment choice or too high of `lo_proj_thresh_active` '
                  '(current value: %s).', mlno.lo_proj_thresh_active)
        raise RuntimeError
    log.info('LO occ proj: %d active | %d standby | %d orthogonal',
             *[u.shape[1] for u in [uocc_loc,uocc_std,uocc_orth]])

    uvir_loc = reduce(jnp.dot, (orbloc.T.conj(), s1e, orbvir))
    uvir_loc, uvir_std, uvir_orth = \
            projection_construction_gpu(uvir_loc, mlno.lo_proj_thresh, mlno.lo_proj_thresh_active)
    log.info('LO vir proj: %d active | %d standby | %d orthogonal',
             *[u.shape[1] for u in [uvir_loc,uvir_std,uvir_orth]])
    if uvir_loc.shape[1] == 0:
        uvir_loc = uvir_std = uvir_orth = None

    ''' LNO construction
    '''
    dmoo = mlno.make_lo_rdm1_occ(eris, moeocc, moevir, uocc_loc, uvir_loc, lno_type[0])
    if mlno._match_oldcode: dmoo *= 0.5 # TO MATCH OLD LNO CODE
    dmoo = reduce(jnp.dot, (uocc_orth.T.conj(), dmoo, uocc_orth))
    if lno_param[0]['norb'] is not None:
        lno_param[0]['norb'] -= uocc_loc.shape[1] + uocc_std.shape[1]
    uoccact_orth, uoccfrz_orth = natorb_select_gpu(dmoo, uocc_orth, **lno_param[0])
    orboccfrz = jnp.hstack((orboccfrz_core, jnp.dot(orbocc, uoccfrz_orth)))
    uoccact = subspace_eigh_gpu(jnp.diag(moeocc), jnp.hstack((uoccact_orth, uocc_std, uocc_loc)))[1]
    orboccact = jnp.dot(orbocc, uoccact)
    uoccact_loc = jnp.linalg.multi_dot((orboccact.T.conj(), s1e, orbloc))
    cput1 = log.timer_debug1('make_lo_rdm1_occ', *cput1)

    dmvv = mlno.make_lo_rdm1_vir(eris, moeocc, moevir, uocc_loc, uvir_loc, lno_type[1])
    if mlno._match_oldcode: dmvv *= 0.5 # TO MATCH OLD LNO CODE
    if uvir_orth is not None:
        dmvv = reduce(np.dot, (uvir_orth.T.conj(), dmvv, uvir_orth))
        if lno_param[1]['norb'] is not None:
            lno_param[1]['norb'] -= uvir_loc.shape[1] + uvir_std.shape[1]
        uviract_orth, uvirfrz_orth = natorb_select_gpu(dmvv, uvir_orth, **lno_param[1])
        orbvirfrz = jnp.hstack((jnp.dot(orbvir, uvirfrz_orth), orbvirfrz_core))
        uviract = subspace_eigh_gpu(jnp.diag(moevir), jnp.hstack((uviract_orth, uvir_std, uvir_loc)))[1]
        orbviract = jnp.dot(orbvir, uviract)
    else:
        orbviract, orbvirfrz = natorb_select_gpu(dmvv, orbvir, **lno_param[1])
        orbvirfrz = jnp.hstack((orbvirfrz, orbvirfrz_core))
        uviract = reduce(jnp.dot, (orbvir.T.conj(), s1e, orbviract))
        uviract = subspace_eigh_gpu(jnp.diag(moevir), uviract)[1]
        orbviract = jnp.dot(orbvir, uviract)
    cput1 = log.timer_debug1('make_lo_rdm1_vir', *cput1)

    ''' LAS construction
    '''
    orbfragall = [orboccfrz, orboccact, orbviract, orbvirfrz]
    orbfrag = np.hstack(orbfragall)
    norbfragall = np.asarray([x.shape[1] for x in orbfragall])
    locfragall = np.cumsum([0] + norbfragall.tolist()).astype(int)
    frzfrag = np.concatenate((
        np.arange(locfragall[0], locfragall[1]),
        np.arange(locfragall[3], locfragall[4]))).astype(int)
    frag_msg = '%d/%d Occ | %d/%d Vir | %d/%d MOs' % (
                    norbfragall[1], sum(norbfragall[:2]),
                    norbfragall[2], sum(norbfragall[2:4]),
                    sum(norbfragall[1:3]), sum(norbfragall)
                )
    if len(frzfrag) == 0:
        frzfrag = 0

    return orbfrag, frzfrag, uoccact_loc, frag_msg

In [89]:
options = {
           'eql_time': 10,
           'n_blocks': 50,
           'n_walkers': 300,
           'mix_precision': True,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'upt2ccsd',
           }

mf = mf
lo_coeff = iao_coeff 
frag_lolist = iao_frag_list
nfrozen = mycc.frozen
thresh = 1e-5
qmc_options = options
chol_cut = 1e-5 
target_sto_error = 1e-3
run_frag_list = [0]
atom_group = None
plot_las = False

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

if spin_type == "unrestricted":
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mf = mlno._scf
else:
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)

mlno.lno_thresh = [thresh*10, thresh]
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag_tot = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag_tot)

frag_lolist = [frag_lolist[i] for i in run_frag_list]
nfrag_run = len(frag_lolist)
print(f"Run {nfrag_run} fragments out of {nfrag_tot} total fragments")

lno_pct_occ = [None, None]
lno_norb = [[None, None]] * nfrag_run

seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                        shape=(nfrag_run,), 
                        minval=0, 
                        maxval=100*nfrag_run
                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag_run
las_size = [None]*nfrag_run
# las_size = np.zeros(nfrag, dtype='int32')
lno_emp2 = np.zeros(nfrag_run, dtype='float64')
lno_ecc  = np.zeros(nfrag_run, dtype='float64')
lno_eqmc = np.zeros(nfrag_run, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')
ccsd_time = np.zeros(nfrag_run, dtype='float64')
qmc_time = np.zeros(nfrag_run, dtype='float64')

mol = mf.mol
print(lno_thresh)
print(lno_pct_occ)
print(lno_norb)

# Loop over fragment
for ifrag, loidx in enumerate(frag_lolist):
    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {run_frag_list[ifrag]+1}/({nfrag_run},{nfrag_tot}) "
    print(msg.center(width, '='))
    if atom_group is not None:
        loc_ctr = f"{atom_group[run_frag_list[ifrag]]}"
        print(f"Center Atom {loc_ctr}")
    else:
        loc_ctr = None
    
    print(f"PySCF NumPy Threads = {lib.num_threads()}")

    if spin_type == "unrestricted":
        orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
        lno_param = [
            [
                {
                    'thresh': (
                        lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                        else lno_thresh[i]
                    ),
                    'pct_occ': (
                        lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                        else lno_pct_occ[i]
                    ),
                    'norb': (
                        lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                        else lno_norb[ifrag][i]
                    ),
                } for i in [0, 1]
            ] for s in range(2)
        ]
    else:
        orbloc = lo_coeff[:,loidx]
        lno_param = [{
            'thresh': lno_thresh[i],
            'pct_occ': lno_pct_occ[i],
            'norb': lno_norb[ifrag][i]
            } for i in [0,1]]

    # M = <orbloc|canactocc> (M^dagger M)u = eu
    # u|canactocc> => orbtial in/out the space spanned by |orbloc>
    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = make_las(mlno, eris, orbloc, lno_type, lno_param)

    mo_occ = mlno.mo_occ

    if spin_type == "unrestricted":
        if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
            lno_elec_type = 'alpha'
        elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
            lno_elec_type = 'beta'
        else:
            lno_elec_type = 'mixed'
        print(f'LNO-Frgament Spin Type = {lno_elec_type}')

        if loc_ctr is None:
            ao_max_a = prep.ao_comp(mf, orbloc[0])
            ao_max_b = prep.ao_comp(mf, orbloc[1])
            loc_ctr = ao_max_a + ao_max_b
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
        occidxa = mo_occ[0] > 1e-10
        occidxb = mo_occ[1] > 1e-10
        moidxa, moidxb = maskact
        nactocc_a = int(np.sum(moidxa & occidxa))
        nactvir_a = int(np.sum(moidxa & ~occidxa))
        nactocc_b = int(np.sum(moidxb & occidxb))
        nactvir_b = int(np.sum(moidxb & ~occidxb))
        nactocc = [nactocc_a, nactocc_b]
        nactvir = [nactvir_a, nactvir_b]
        lno_active_a = np.array([i for i in range(mol.nao) if i not in lno_frozen[0]])
        lno_active_b = np.array([i for i in range(mol.nao) if i not in lno_frozen[1]])
        lno_active = [lno_active_a, lno_active_b]
        lno_tot = [len(lno_active_a), len(lno_active_b)]
        # print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    else:
        print(f'LNO-Frgament Spin Type = restricted')
        if loc_ctr is None:
            loc_ctr = prep.ao_comp(mf, orbloc)
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
        lno_active = np.array([i for i in range(mol.nao) if i not in lno_frozen])
        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        lno_tot = len(lno_active)
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    
    if plot_las:
        tools.plot_density(mol, orbloc, lno_coeff, lno_active, spin_type, idx = run_frag_list[ifrag]+1)

    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 =\
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
    time1 = time.perf_counter()
    lnocc_time = time1 - time0

    print(f'LNO-MP2 Orbital Energy:  {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')
    print(f"LNO-CCSD time:           {lnocc_time:.2f} s")

Run 1 fragments out of 6 total fragments
[0.0001, 1e-05]
[None, None]
[[None, None]]


======================= restricted LNO-FRAGMENT 1/(1,6) ========================
PySCF NumPy Threads = 16
LNO-Frgament Spin Type = restricted
LNO Center 0 O 1s    
LAS occupied orbitals:  7
LAS virtual orbitals:   28
LAS total size:         35
LNO-MP2 Orbital Energy:  -0.17003705
LNO-CCSD Orbital Energy: -0.17752155
LNO-CCSD time:           0.27 s


In [90]:
options = {
           'eql_time': 10,
           'n_blocks': 50,
           'n_walkers': 300,
           'mix_precision': True,
           'seed': 17,
           'walker_type': 'uhf',
           'trial': 'upt2ccsd',
           }

mf = mf
lo_coeff = iao_coeff 
frag_lolist = iao_frag_list
nfrozen = mycc.frozen
thresh = 1e-5
qmc_options = options
chol_cut = 1e-5 
target_sto_error = 1e-3
run_frag_list = [0]
atom_group = None
plot_las = False

spin_type = prep.kind(lo_coeff)

if frag_lolist is None:
    if spin_type == "unrestricted":
        raise ValueError("frag_lolist must be provided for unrestricted LNO-AFQMC.")
    print("Fragment list not found. Asign every LO to a fragment.")
    frag_lolist = [[i] for i in range(lo_coeff.shape[1])]

if spin_type == "unrestricted":
    mlno = ulnoccsd.ULNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)
    mf = mlno._scf
else:
    mlno = lnoccsd.LNOCCSD(mf, lo_coeff, frag_lolist, frozen=nfrozen).set(verbose=3)

mlno.lno_thresh = [thresh*10, thresh]
lno_thresh = mlno.lno_thresh
lno_type = ['1h','1h']
eris = mlno.ao2mo()

nfrag_tot = len(frag_lolist)
if run_frag_list is None:
    run_frag_list = range(nfrag_tot)

frag_lolist = [frag_lolist[i] for i in run_frag_list]
nfrag_run = len(frag_lolist)
print(f"Run {nfrag_run} fragments out of {nfrag_tot} total fragments")

lno_pct_occ = [None, None]
lno_norb = [[None, None]] * nfrag_run

seeds = random.randint(random.PRNGKey(qmc_options["seed"]),
                        shape=(nfrag_run,), 
                        minval=0, 
                        maxval=100*nfrag_run
                        )

qmc_options["max_error"] = target_sto_error / np.sqrt(nfrag_tot)
trial_base = qmc_options.get("trial", "")

las_center = [None]*nfrag_run
las_size = [None]*nfrag_run
# las_size = np.zeros(nfrag, dtype='int32')
lno_emp2 = np.zeros(nfrag_run, dtype='float64')
lno_ecc  = np.zeros(nfrag_run, dtype='float64')
lno_eqmc = np.zeros(nfrag_run, dtype='float64')
lno_eqmc_err  = np.zeros(nfrag_run, dtype='float64')
ccsd_time = np.zeros(nfrag_run, dtype='float64')
qmc_time = np.zeros(nfrag_run, dtype='float64')

mol = mf.mol
print(lno_thresh)
print(lno_pct_occ)
print(lno_norb)

# Loop over fragment
for ifrag, loidx in enumerate(frag_lolist):
    print("\n")
    width = 80
    msg = f" {spin_type} LNO-FRAGMENT {run_frag_list[ifrag]+1}/({nfrag_run},{nfrag_tot}) "
    print(msg.center(width, '='))
    if atom_group is not None:
        loc_ctr = f"{atom_group[run_frag_list[ifrag]]}"
        print(f"Center Atom {loc_ctr}")
    else:
        loc_ctr = None
    
    print(f"PySCF NumPy Threads = {lib.num_threads()}")

    if spin_type == "unrestricted":
        orbloc = [lo_coeff[0][:,loidx[0]], lo_coeff[1][:,loidx[1]]]
        lno_param = [
            [
                {
                    'thresh': (
                        lno_thresh[i][s] if isinstance(lno_thresh[i], Iterable)
                        else lno_thresh[i]
                    ),
                    'pct_occ': (
                        lno_pct_occ[i][s] if isinstance(lno_pct_occ[i], Iterable)
                        else lno_pct_occ[i]
                    ),
                    'norb': (
                        lno_norb[ifrag][i][s] if isinstance(lno_norb[ifrag][i], Iterable)
                        else lno_norb[ifrag][i]
                    ),
                } for i in [0, 1]
            ] for s in range(2)
        ]
    else:
        orbloc = lo_coeff[:,loidx]
        lno_param = [{
            'thresh': lno_thresh[i],
            'pct_occ': lno_pct_occ[i],
            'norb': lno_norb[ifrag][i]
            } for i in [0,1]]

    # M = <orbloc|canactocc> (M^dagger M)u = eu
    # u|canactocc> => orbtial in/out the space spanned by |orbloc>
    # uocc_loc = <lno_actocc|orbloc>
    lno_coeff, lno_frozen, uocc_loc, _ = make_las_gpu(mlno, eris, orbloc, lno_type, lno_param)

    mo_occ = mlno.mo_occ

    if spin_type == "unrestricted":
        if uocc_loc[0].size > 0 and uocc_loc[1].size == 0:
            lno_elec_type = 'alpha'
        elif uocc_loc[0].size == 0 and uocc_loc[1].size > 0:
            lno_elec_type = 'beta'
        else:
            lno_elec_type = 'mixed'
        print(f'LNO-Frgament Spin Type = {lno_elec_type}')

        if loc_ctr is None:
            ao_max_a = prep.ao_comp(mf, orbloc[0])
            ao_max_b = prep.ao_comp(mf, orbloc[1])
            loc_ctr = ao_max_a + ao_max_b
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = ulnoccsd.get_maskact(lno_frozen, [mo_occ[0].size, mo_occ[1].size])
        occidxa = mo_occ[0] > 1e-10
        occidxb = mo_occ[1] > 1e-10
        moidxa, moidxb = maskact
        nactocc_a = int(np.sum(moidxa & occidxa))
        nactvir_a = int(np.sum(moidxa & ~occidxa))
        nactocc_b = int(np.sum(moidxb & occidxb))
        nactvir_b = int(np.sum(moidxb & ~occidxb))
        nactocc = [nactocc_a, nactocc_b]
        nactvir = [nactvir_a, nactvir_b]
        lno_active_a = np.array([i for i in range(mol.nao) if i not in lno_frozen[0]])
        lno_active_b = np.array([i for i in range(mol.nao) if i not in lno_frozen[1]])
        lno_active = [lno_active_a, lno_active_b]
        lno_tot = [len(lno_active_a), len(lno_active_b)]
        # print(f'LAS alpha: {nactocc_a} occupied, {nactvir_a} virtual')
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = ulnoccsd.UCCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    else:
        print(f'LNO-Frgament Spin Type = restricted')
        if loc_ctr is None:
            loc_ctr = prep.ao_comp(mf, orbloc)
            print(f"LNO Center {loc_ctr}")

        lno_frozen, maskact = lnoccsd.get_maskact(lno_frozen, mo_occ.size)
        lno_active = np.array([i for i in range(mol.nao) if i not in lno_frozen])
        nactocc, nactvir = prep.las_size(mf, lno_frozen)
        lno_tot = len(lno_active)
        print(f'LAS occupied orbitals:  {nactocc}')
        print(f'LAS virtual orbitals:   {nactvir}')
        print(f'LAS total size:         {lno_tot}')
        mcc = lnoccsd.CCSD(mf, mo_coeff=lno_coeff, frozen=lno_frozen).set(verbose=1)
    
    if plot_las:
        tools.plot_density(mol, orbloc, lno_coeff, lno_active, spin_type, idx = run_frag_list[ifrag]+1)

    mcc._s1e = mlno._s1e
    mcc._h1e = mlno._h1e
    mcc._vhf = mlno._vhf
    if mlno.kwargs_imp is not None:
        mcc = mcc.set(**mlno.kwargs_imp)
    time0 = time.perf_counter()
    (eorb_mp2, eorb_cc), t1, t2 =\
        mod_lnoccsd.lnoccsd_kernel(mcc, lno_coeff, uocc_loc, mo_occ, maskact)
    time1 = time.perf_counter()
    lnocc_time = time1 - time0

    print(f'LNO-MP2 Orbital Energy:  {eorb_mp2:.8f}')
    print(f'LNO-CCSD Orbital Energy: {eorb_cc:.8f}')
    print(f"LNO-CCSD time:           {lnocc_time:.2f} s")

Run 1 fragments out of 6 total fragments
[0.0001, 1e-05]
[None, None]
[[None, None]]


======================= restricted LNO-FRAGMENT 1/(1,6) ========================
PySCF NumPy Threads = 16
LNO-Frgament Spin Type = restricted
LNO Center 0 O 1s    
LAS occupied orbitals:  7
LAS virtual orbitals:   28
LAS total size:         35


E0614 18:35:46.213941   61686 cuda_executor.cc:1213] [0] Failed to track device allocation: ALREADY_EXISTS: Allocation at address (nil) (size 0) is already tracked


LNO-MP2 Orbital Energy:  -0.17003705
LNO-CCSD Orbital Energy: -0.17752155
LNO-CCSD time:           0.25 s
